# 10. Xử lý mất cân bằng lớp nâng cao (Phase 9)

Notebook triển khai ba thí nghiệm mở rộng trên cùng **backbone C** (`fusion_concat_frozen_reg`) và cùng giao thức với notebook `09_imbalance.ipynb`:

| ID | Phương pháp | Cấu hình | Mục đích |
|---|---|---|---|
| B1 | CE + class weight `sqrt_inverse` | `fusion_concat_frozen_reg_cw` | Đối chứng tốt nhất hiện tại |
| B6 | Logit Adjustment, τ=1 | `fusion_concat_frozen_reg_logit_adjustment` | Điều chỉnh biên quyết định trong lúc train |
| B7 | Minority-aware Manifold Mixup, α=0.2 | `fusion_concat_frozen_reg_mixup` | Nội suy đặc trưng sau fusion |
| B8 | Post-hoc class-wise calibration | Dùng lại checkpoint B1 | Tối ưu quyết định, không train lại |

**Nguyên tắc:** split chính thức, seed 42, cùng số bước tối ưu, chọn checkpoint theo macro-F1 bỏ `text-sarcasm`, không chấm private test khi screening. B6/B7 so với B1. B8 học hệ số trên **OOF của train**, sau đó đánh giá trên **val**; không tối ưu và báo cáo trên cùng một tập.

**Kaggle Input:** `phuclkz/vimmsd-private-dataset`, `vtnmai/vimmsd-data-preprocessed`. Có thể Add output notebook Phase 9 trước để dùng lại B1. B8 cần thêm output của plan `empirical_C_screen` chứa `empC_B1_f{0,1,2}_s42/val_predictions.csv`.

**Settings:** GPU T4 x2, Internet on. Đổi `REPO`/`REF` ở cell bootstrap nếu chạy fork khác; branch phải được push trước khi clone trên Kaggle.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = "github.com/PhucLe110/Multimodal_Sarcasm_Detection.git"
REF = "feat/phase9-imbalance-advanced"
CODE_DIR = "/tmp/IE403"

def get_github_token():
    try:
        if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("GITHUB_TOKEN")
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None

ON_CLOUD = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or "COLAB_RELEASE_TAG" in os.environ
if ON_CLOUD:
    token = get_github_token()
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    subprocess.run(["rm", "-rf", CODE_DIR])
    subprocess.run(["git", "clone", "-q", url, CODE_DIR], check=True)
    subprocess.run(["git", "-C", CODE_DIR, "checkout", "-q", REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE_DIR}/requirements-kaggle.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE_DIR], check=True)
    if CODE_DIR + "/src" not in sys.path:
        sys.path.insert(0, CODE_DIR + "/src")
    os.chdir(CODE_DIR)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

print("cwd:", os.getcwd())
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code commit:", commit or "unknown")

In [ ]:
RUN_TESTS = True
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
    result = subprocess.run([sys.executable, "-m", "pytest", "tests/test_imbalance.py", "-q", "-x", "-p", "no:cacheprovider"], capture_output=True, text=True)
    print(result.stdout[-4000:], result.stderr[-2000:])
    if result.returncode != 0:
        raise RuntimeError("Test B6–B8 thất bại; dừng trước khi train")

In [ ]:
import json, logging, re, shutil
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display

from vimmsd.utils.config import load_config
from vimmsd.utils.experiments import expand_runs, load_plan, run_output_dir

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)
pd.set_option("display.max_colwidth", 120)

PLAN = "configs/experiments/empirical_imbalance_advanced.yaml"
PREVIOUS_OUTPUT = None  # Output Phase 9 cũ để tái sử dụng B1; None = tự tìm trong /kaggle/input
OOF_DIR = None          # Thư mục chứa empC_B1_f{0,1,2}_s42; None = tự tìm khi chạy B8

METHODS = {
    "B1": ("emp_c_cw", "Class weight √"),
    "B6": ("emp_c_logit_adjust", "Logit Adjustment"),
    "B7": ("emp_c_mixup", "Manifold Mixup"),
}
plan = load_plan(PLAN)
specs = {s.group: s for s in expand_runs(plan)}
OUT = run_output_dir(next(iter(specs.values()))).parent
cfg = load_config(specs["emp_c_cw"].config, specs["emp_c_cw"].overrides)
LABELS = list(cfg.data.labels)
RARE = "text-sarcasm"
print("output:", OUT)

## 1. Kiểm tra dữ liệu và đối chứng

Ba phương pháp mới chỉ thay **loss hoặc phép biến đổi đặc trưng**. Không đổi encoder, tập train, sampler, số lượt rút mỗi epoch hay tiêu chí chọn checkpoint. Vì vậy chênh lệch với B1 có thể gắn với chính kỹ thuật đang thử.

In [ ]:
from vimmsd.data.dataset import load_all_records, stage_preprocessed_cache
from vimmsd.data.preprocessing import TextPreprocessor

data_dir = Path(cfg.paths.data_dir)
if not (data_dir / cfg.data.train_json).exists():
    raise FileNotFoundError(f"Không thấy {data_dir / cfg.data.train_json}; hãy Add Input phuclkz/vimmsd-private-dataset")
expected_key = TextPreprocessor.from_config(cfg.data.text).pipeline_key()
for path in stage_preprocessed_cache(cfg):
    key = json.loads(path.read_text(encoding="utf-8")).get("meta", {}).get("pipeline_key")
    print(f"{path.name}: pipeline_key={key}, code hiện tại={expected_key}")
    if key != expected_key:
        raise ValueError("Cache khác pipeline tiền xử lý; hãy Add đúng vtnmai/vimmsd-data-preprocessed")

splits = load_all_records(cfg)
counts = pd.DataFrame({split: Counter(LABELS[r["label"]] for r in splits[split]) for split in ("train", "val")}).reindex(LABELS)
counts["train (%)"] = (100 * counts["train"] / counts["train"].sum()).round(2)
display(counts)
assert (len(splits["train"]), len(splits["val"])) == (10805, 1413)

## 2. B6 — Logit Adjustment / Balanced Softmax

Với `π_c = N_c / ΣN`, loss cộng độ dịch tần suất vào logit **chỉ trong lúc train**:

$$L_{LA} = CE\big(f(x) + \tau\log\pi,\ y\big).$$

Mô hình phải học logit thô lớn hơn cho lớp hiếm để bù lượng `τ·log(π_c)` âm mạnh. Khi dự đoán, dùng **logit thô**, không cộng lại độ dịch. Đây là điểm khác với hiệu chỉnh prior sau train.

- `tau: 1.0`: thiết lập chuẩn để screening; không dùng đồng thồi class weight.
- Nếu B6 làm recall lớp hiếm tăng quá mạnh và precision sụt, có thể chạy thêm τ=0.5 như một thí nghiệm phụ, không chọn τ dựa trên private test.
- Code: `LogitAdjustmentLoss` trong `src/vimmsd/training/losses.py`; config `configs/fusion_concat_frozen_reg_logit_adjustment.yaml`.

In [ ]:
from vimmsd.training.losses import LogitAdjustmentLoss

b6_cfg = load_config(specs["emp_c_logit_adjust"].config, specs["emp_c_logit_adjust"].overrides)
class_counts = [int(counts.loc[label, "train"]) for label in LABELS]
priors = np.array(class_counts) / np.sum(class_counts)
display(pd.DataFrame({"lớp": LABELS, "N_c": class_counts, "prior": priors, "τ·log(prior)": b6_cfg.loss.tau * np.log(priors)}))

# Ví dụ số: loss nhận logit thô và tự cộng shift trước CE.
torch.manual_seed(42)
demo_logits = torch.randn(4, 4)
demo_y = torch.arange(4)
b6_loss = LogitAdjustmentLoss(class_counts, tau=b6_cfg.loss.tau)
manual = F.cross_entropy(demo_logits + b6_cfg.loss.tau * torch.log(torch.tensor(priors, dtype=demo_logits.dtype)), demo_y)
print("Loss implementation:", float(b6_loss(demo_logits, demo_y)))
print("Loss công thức thủ công:", float(manual))
assert torch.isclose(b6_loss(demo_logits, demo_y), manual)

## 3. B7 — Minority-aware Manifold Mixup sau fusion

Với vector fusion `z = [text_proj(caption); image_proj(CLIP)]`, chọn hoán vị `j` trong cùng mini-batch và trộn:

$$\tilde z = \lambda z_i + (1-\lambda)z_j,\qquad \tilde y = \lambda y_i + (1-\lambda)y_j,\qquad \lambda\sim Beta(\alpha,\alpha).$$

Head MLP nhận `ẑ`, loss dùng **soft target** `ỹ`. Không trộn logit và không trộn ảnh/caption thô.

Để ưu tiên lớp hiếm, nếu anchor `i` là `text-sarcasm` hoặc `image-sarcasm` và λ<0.5, code đổi λ thành `1−λ`; anchor hiếm luôn giữ ít nhất 50% đặc trưng. Cách này không làm tăng số lần mẫu hiếm xuất hiện như oversampling.

- `mixup_alpha: 0.2`: α nhỏ tạo nhiều phép trộn gần một trong hai điểm gốc.
- B7 dùng flat head + CE, không class weight và không loss phụ để tránh lẫn tác dụng.
- Code: `manifold_mixup` trong `models/fusion.py`, `SarcasmClassifier.forward_mixup`, `ManifoldMixupLoss` và nhánh `uses_mixup` trong `training/trainer.py`.

In [ ]:
from vimmsd.models.fusion import manifold_mixup

b7_cfg = load_config(specs["emp_c_mixup"].config, specs["emp_c_mixup"].overrides)
minority_ids = tuple(LABELS.index(x) for x in b7_cfg.loss.mixup_minority_labels)
torch.manual_seed(7)
z = torch.randn(8, 6)
y = torch.tensor([0, 1, 0, 2, 3, 0, 2, 3])
z_mix, lam, perm = manifold_mixup(z, y, alpha=b7_cfg.loss.mixup_alpha, minority_labels=minority_ids)
demo = pd.DataFrame({"anchor": y.tolist(), "partner": y[perm].tolist(), "lambda": lam.squeeze(1).numpy()})
demo["anchor hiếm"] = demo["anchor"].isin(minority_ids)
display(demo)
assert (lam[demo["anchor hiếm"].to_numpy()] >= 0.5).all()
assert torch.allclose(z_mix, lam * z + (1 - lam) * z[perm])

# Target đầu vào loss là một vector xác suất mềm, không phải một nhãn cứng.
one_hot = F.one_hot(y, 4).float()
soft_target = lam * one_hot + (1 - lam) * one_hot[perm]
print("Ví dụ soft target của anchor đầu:", soft_target[0])

## 4. Train B6 và B7

Plan gồm B1, B6, B7 và chạy `evaluate.py --split val` sau train để lưu xác suất. Nếu output B1 cũ có đủ checkpoint, metrics và predictions thì notebook tái sử dụng; nếu thiếu, B1 được train lại để B8 có cùng giao thức.

Kiểm tra `run.log` của từng run: loss giảm, không NaN, số lượt rút giống B1. Không bật `train.eval_test` trong bước này.

In [ ]:
if PREVIOUS_OUTPUT is None and Path("/kaggle/input").exists():
    hits = sorted(Path("/kaggle/input").rglob("emp_c_cw_s42/results_row.md"))
    PREVIOUS_OUTPUT = hits[0].parent.parent if hits else None

if PREVIOUS_OUTPUT:
    src = Path(PREVIOUS_OUTPUT) / "emp_c_cw_s42"
    required = ["best.pt", "best_val_metrics.json", "val_predictions.csv", "results_row.md"]
    if all((src / f).exists() for f in required) and src.resolve() != (OUT / src.name).resolve():
        shutil.copytree(src, OUT / src.name, dirs_exist_ok=True)
        print("Đã tái sử dụng B1 từ", src)
    else:
        print("B1 cũ thiếu checkpoint/predictions; plan sẽ train lại B1")

subprocess.run([sys.executable, "scripts/run_experiments.py", "--plan", PLAN, "--dry-run"], check=True)

In [ ]:
!python scripts/run_experiments.py --plan {PLAN}

## 5. So sánh B1, B6, B7

Chỉ số xếp hạng là **val macro-F1 4 lớp**; macro-F1 bỏ `text-sarcasm` là tiêu chí chọn checkpoint vì val chỉ có một mẫu text. Luôn đọc thêm F1/recall từng lớp: tăng recall lớp hiếm nhưng giảm mạnh precision hoặc lớp `not-sarcasm` là một đánh đổi, chưa chắc là cải thiện.

In [ ]:
def read_json(path):
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None

def metric_row(mid, desc, m):
    return {"ID": mid, "kỹ thuật": desc, "macro-F1": m["macro_f1"],
            "macro-F1 bỏ text": np.mean([v["f1"] for c, v in m["per_class"].items() if c != RARE]),
            "weighted-F1": m["weighted_f1"], "accuracy": m["accuracy"]}

metrics = {}
rows, class_rows = [], []
for mid, (group, desc) in METHODS.items():
    m = read_json(OUT / f"{group}_s42" / "best_val_metrics.json")
    if m is None:
        print(mid, "chưa có kết quả")
        continue
    metrics[mid] = m
    rows.append(metric_row(mid, desc, m))
    for label, values in m["per_class"].items():
        class_rows.append({"ID": mid, "lớp": label, **{k: values[k] for k in ("precision", "recall", "f1", "support")}})

table = pd.DataFrame(rows).set_index("ID")
if "B1" in table.index:
    table["Δ vs B1"] = table["macro-F1"] - table.loc["B1", "macro-F1"]
display(table.style.format(precision=4).highlight_max(subset=["macro-F1"], color="#c6efce"))
per_class = pd.DataFrame(class_rows)
display(per_class.pivot(index="ID", columns="lớp", values=["precision", "recall", "f1"]).round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
table["macro-F1"].plot.bar(ax=axes[0], color="#4c72b0", rot=0)
axes[0].set_title("Val macro-F1 (seed 42)"); axes[0].set_ylim(0, 1)
per_class.pivot(index="lớp", columns="ID", values="f1").reindex(LABELS).plot.bar(ax=axes[1], rot=0)
axes[1].set_title("F1 từng lớp"); axes[1].set_ylim(0, 1); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 6. B8 — Post-hoc Class-wise Calibration

Giữ nguyên checkpoint B1 và xác suất `P(y=c|x)`. Tìm hệ số `α=(α_0,…,α_3)` để dự đoán:

$$\hat y = \arg\max_c\; \alpha_c P(y=c|x).$$

`α_not = 1` làm lớp tham chiếu vì nhân toàn bộ α với cùng một hằng số không đổi argmax. Ba hệ số còn lại được tối ưu trên log-scale trong `[0.1, 10]` bằng differential evolution. Macro-F1 là hàm không liên tục theo α nên không dùng gradient descent.

**Chống leakage:**
1. Nối `val_predictions.csv` của ba fold `empC_B1_f0/f1/f2`: đây là dự đoán OOF trên 10.805 mẫu train, có 77 mẫu text và 442 mẫu image.
2. Chỉ dùng OOF để học α. Điểm trên OOF sau tối ưu là điểm fitting, **không dùng để kết luận cải thiện**.
3. Áp α lên `val_predictions.csv` của B1 train trên full train. Val không tham gia tối ưu α.
4. Không thay bằng cách tối ưu α trên chính val (chỉ 1 text, 28 image) rồi báo cáo lại val.

B8 không hứa chắc tăng 2–5%. Nếu α quá lớn cho text/image hoặc val giảm, ghi nhận calibration không ổn định và giữ B1.

In [ ]:
if OOF_DIR is None and Path("/kaggle/input").exists():
    hits = sorted(Path("/kaggle/input").rglob("empC_B1_f0_s42/val_predictions.csv"))
    OOF_DIR = hits[0].parent.parent if hits else None

oof_paths = [Path(OOF_DIR) / f"empC_B1_f{i}_s42" / "val_predictions.csv" for i in range(3)] if OOF_DIR else []
eval_path = OUT / "emp_c_cw_s42" / "val_predictions.csv"
missing = [str(p) for p in [*oof_paths, eval_path] if not p.exists()]
if len(oof_paths) != 3 or missing:
    raise FileNotFoundError(
        "B8 cần 3 file OOF empC_B1_f{0,1,2}_s42/val_predictions.csv và file val của emp_c_cw_s42. "
        "Add output plan empirical_C_screen, đặt OOF_DIR ở cell setup rồi chạy lại. Thiếu: " + str(missing))
print("OOF:", *oof_paths, sep="\n  ")
print("Evaluation:", eval_path)

In [ ]:
prob_cols = [f"prob_{label}" for label in LABELS]

def load_predictions(paths):
    if isinstance(paths, (str, Path)):
        paths = [paths]
    df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
    assert df["id"].is_unique, "ID bị trùng khi ghép OOF"
    y = df["label"].map({label: i for i, label in enumerate(LABELS)}).to_numpy(dtype=int)
    p = df[prob_cols].to_numpy(dtype=float)
    assert np.allclose(p.sum(axis=1), 1.0, atol=1e-3)
    return df, p, y

cal_df, cal_p, cal_y = load_predictions(oof_paths)
val_df, val_p, val_y = load_predictions(eval_path)
print("Calibration OOF:", cal_p.shape, Counter(cal_y))
print("Evaluation val:", val_p.shape, Counter(val_y))
assert len(cal_df) == 10805 and len(val_df) == 1413

In [ ]:
from vimmsd.analysis.calibration import apply_class_multipliers, tune_class_multipliers
from vimmsd.training.metrics import compute_metrics

alpha, fit_info = tune_class_multipliers(cal_p, cal_y, bounds=(0.1, 10.0), seed=42, maxiter=80, popsize=12)
display(pd.DataFrame({"lớp": LABELS, "alpha": alpha}))
display(pd.Series(fit_info, name="giá trị"))
assert alpha[0] == 1.0 and fit_info["calibration_macro_f1"] >= fit_info["baseline_macro_f1"]

In [ ]:
calibrated_p = apply_class_multipliers(val_p, alpha, normalize=True)
before = compute_metrics(val_y, val_p.argmax(axis=1), LABELS)
after = compute_metrics(val_y, calibrated_p.argmax(axis=1), LABELS)
metrics["B8"] = after

b8_table = pd.DataFrame([metric_row("B1", "Trước calibration", before), metric_row("B8", "Sau calibration", after)]).set_index("ID")
b8_table["Δ"] = b8_table["macro-F1"] - b8_table.loc["B1", "macro-F1"]
display(b8_table.style.format(precision=4))

b8_classes = []
for mid, m in (("B1", before), ("B8", after)):
    for label, values in m["per_class"].items():
        b8_classes.append({"ID": mid, "lớp": label, **{k: values[k] for k in ("precision", "recall", "f1")}})
b8_classes = pd.DataFrame(b8_classes)
display(b8_classes.pivot(index="ID", columns="lớp", values=["precision", "recall", "f1"]).round(4))

b8_dir = OUT / "emp_c_cw_calibrated_s42"
b8_dir.mkdir(parents=True, exist_ok=True)
(b8_dir / "calibration.json").write_text(json.dumps({"labels": LABELS, "alpha": alpha.tolist(), "fit_oof": fit_info}, ensure_ascii=False, indent=2))
(b8_dir / "val_metrics.json").write_text(json.dumps(after, indent=2))
saved = val_df.copy()
saved["pred"] = [LABELS[i] for i in calibrated_p.argmax(axis=1)]
saved[prob_cols] = calibrated_p
saved.to_csv(b8_dir / "val_predictions.csv", index=False)
print("Đã lưu B8 tại", b8_dir)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, metric in zip(axes, ("f1", "recall")):
    b8_classes.pivot(index="lớp", columns="ID", values=metric).reindex(LABELS).plot.bar(ax=ax, rot=0)
    ax.set_title(f"{metric} trước/sau calibration"); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

## 7. Tổng hợp và quy tắc đọc kết quả

1. **B6/B7:** chỉ giữ ứng viên vượt B1 về macro-F1; kiểm tra thêm F1 từng lớp và đường train.
2. **B8:** so trên val sau khi α đã được cố định bằng OOF; không dùng `fit_oof.calibration_macro_f1` làm bằng chứng tổng quát hóa.
3. Một seed trên val chỉ là screening. Ứng viên tốt cần chạy lại theo 3-fold OOF và/hoặc seeds 1, 2, 42 trước khi kết luận.
4. Không chấm private test trong notebook này. Chỉ chốt phương pháp rồi mới chạy private test một lần theo giao thức chung.
5. Nếu không phương pháp nào vượt B1, giữ B1; không chọn dựa vào weighted-F1/accuracy vì hai chỉ số này bị lớp lớn chi phối.

In [ ]:
desc_all = {**{mid: desc for mid, (_, desc) in METHODS.items()}, "B8": "B1 + calibration"}
final = pd.DataFrame([metric_row(mid, desc_all[mid], m) for mid, m in metrics.items()]).set_index("ID")
final["Δ vs B1"] = final["macro-F1"] - final.loc["B1", "macro-F1"]
display(final.style.format(precision=4).highlight_max(subset=["macro-F1"], color="#c6efce"))

print("| ID | Kỹ thuật | Macro-F1 | Macro-F1 bỏ text | Weighted-F1 | Accuracy | Δ vs B1 |")
print("|---|---|---:|---:|---:|---:|---:|")
for mid, row in final.iterrows():
    print(f"| {mid} | {row['kỹ thuật']} | {row['macro-F1']:.4f} | {row['macro-F1 bỏ text']:.4f} | {row['weighted-F1']:.4f} | {row['accuracy']:.4f} | {row['Δ vs B1']:+.4f} |")